# Perturb-CITE-seq GO importance vs. GSEA/ORA

In [ ]:
mode = "subset"


In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

colors = [
    "#70acc0",
    "#c12075",
    "#003754",
    "#4c626c",
    "#be9e7c",
    "#723c5d",
    "#9d9c9c",
    "#00695c",
    "#e57373",
]
colors_name = [
    "graublau",
    "purpur",
    "blau",
    "dunkelgrau",
    "bronze",
    "violett",
    "hellgrau",
    "gedämpftes teal",
    "sanftes koralle",
]
sns.set_palette(sns.color_palette(colors))
base_color = "#70acc0"
light_palette = sns.light_palette(base_color, as_cmap=False)
colors = light_palette + ["#4c626c", "#003754"]
cmap = LinearSegmentedColormap.from_list("custom_cmap", colors)
colors_heat = ["#c12075", "#f0f0f0", "#003754"]
cmap_heat = LinearSegmentedColormap.from_list("custom_diverging", colors_heat, N=256)
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42
mpl.rcParams["svg.fonttype"] = "none"

import re as _re
import matplotlib.figure as _mpl_figure

def _fix_svg_font_declarations(_svg_path):
    """Convert matplotlib's unsupported CSS 'font:' shorthand (labeled px, but
    actually point-sized, in a pt-calibrated viewBox) into explicit
    font-family/font-size/font-weight/font-style, which is what Figma's SVG
    importer requires to resolve size and style correctly."""
    try:
        with open(_svg_path, "r", encoding="utf-8") as _f:
            _content = _f.read()
    except OSError:
        return
    def _convert(_m):
        _oblique = _m.group(1)
        _weight = _m.group(2)
        _pt = float(_m.group(3)) * 0.75
        _family = _m.group(4)
        _font_weight = "bold" if _weight else "normal"
        _font_style = "oblique" if _oblique else "normal"
        return (
            f"font-family: {_family}; font-size: {_pt:g}pt; "
            f"font-weight: {_font_weight}; font-style: {_font_style}"
        )
    _pattern = _re.compile(r'font:\s*(oblique\s+)?(?:(\d+)\s+)?([0-9.]+)px\s+([^;"]+)')
    _new_content = _pattern.sub(_convert, _content)
    if _new_content != _content:
        with open(_svg_path, "w", encoding="utf-8") as _f:
            _f.write(_new_content)

_orig_figure_savefig = _mpl_figure.Figure.savefig

def _rasterize_dense_elements(_fig):
    for _ax in _fig.get_axes():
        for _coll in list(_ax.collections):
            try:
                _coll.set_rasterized(True)
            except Exception:
                pass
        for _im in _ax.get_images():
            try:
                _im.set_rasterized(True)
            except Exception:
                pass

def _patched_figure_savefig(self, fname, *args, **kwargs):
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _rasterize_dense_elements(self)
        kwargs.setdefault("dpi", 300)
    _result = _orig_figure_savefig(self, fname, *args, **kwargs)
    if isinstance(fname, str) and fname.lower().endswith(".svg"):
        _fix_svg_font_declarations(fname)
    return _result

_mpl_figure.Figure.savefig = _patched_figure_savefig
mpl.rcParams["figure.dpi"] = 500.0
mpl.rcParams["savefig.transparent"] = True
mpl.rcParams["savefig.format"] = "png"

mpl.rcParams["font.family"] = "sans-serif"
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans"]

mpl.rcParams["axes.edgecolor"] = "black"
mpl.rcParams["axes.linewidth"] = 0.8
mpl.rcParams["axes.grid"] = True
mpl.rcParams["grid.color"] = "grey"
mpl.rcParams["grid.linestyle"] = "--"
mpl.rcParams["grid.linewidth"] = 0.5

mpl.rcParams["legend.frameon"] = False
mpl.rcParams["legend.loc"] = "best"

mpl.rcParams["xtick.major.size"] = 3
mpl.rcParams["xtick.minor.size"] = 2
mpl.rcParams["ytick.major.size"] = 3
mpl.rcParams["ytick.minor.size"] = 2
mpl.rcParams["xtick.major.width"] = 0.4
mpl.rcParams["xtick.minor.width"] = 0.3
mpl.rcParams["ytick.major.width"] = 0.4
mpl.rcParams["ytick.minor.width"] = 0.3
scaling_factor = 1
if mode == "all":
    base_font_size = 4
else:
    base_font_size = 6
base_axes_title_size = base_font_size + 1
base_axes_label_size = base_font_size
base_tick_label_size = base_font_size - 0.5
base_legend_font_size = base_font_size - 0.5
base_figure_title_size = base_font_size + 2
mpl.rcParams["font.size"] = base_font_size * scaling_factor
mpl.rcParams["axes.titlesize"] = base_axes_title_size * scaling_factor
mpl.rcParams["axes.labelsize"] = base_axes_label_size * scaling_factor
mpl.rcParams["xtick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["ytick.labelsize"] = base_tick_label_size * scaling_factor
mpl.rcParams["legend.fontsize"] = base_legend_font_size * scaling_factor
mpl.rcParams["figure.titlesize"] = base_figure_title_size * scaling_factor

mpl.rcParams["patch.linewidth"] = 0.7

mpl.rcParams["figure.autolayout"] = True


In [ ]:
def set_figure_params(column, height):

    single_column_width = 3.5
    double_column_width = 7.0

    if column == "single":
        figure_width = single_column_width
    elif column == "double":
        figure_width = double_column_width
    else:
        figure_width = column
    scaling_factor = figure_width / single_column_width
    scaling_factor = 1

    mpl.rcParams["figure.figsize"] = (figure_width, height)


In [ ]:
import sys
import os

_OUTDIR_ABS = os.path.join(os.getcwd(), "outputs", "fig6_S25_S26_frangieh2021_perturbation_analysis")
os.makedirs(_OUTDIR_ABS, exist_ok=True)

sys.path.append(os.path.abspath("../.experiment_data/primary/benchmarking"))

import contextlib
import functools

_POSTPROC_CWD = os.getcwd()

@contextlib.contextmanager
def _in_primary_dir():
    os.chdir(os.path.join(_POSTPROC_CWD, "../.experiment_data/primary"))
    try:
        yield
    finally:
        os.chdir(_POSTPROC_CWD)

with _in_primary_dir():
    import muon as _muon
    _orig_sample_obs = _muon.pp.sample_obs
    def _patched_sample_obs(data, *a, **kw):
        return _orig_sample_obs(data, *a, **kw).copy()
    _muon.pp.sample_obs = _patched_sample_obs
    import datasets

for _name in dir(datasets):
    if _name.startswith("load_"):
        def _make_wrapper(fn):
            @functools.wraps(fn)
            def _wrapper(*a, **kw):
                with _in_primary_dir():
                    return fn(*a, **kw)
            return _wrapper
        setattr(datasets, _name, _make_wrapper(getattr(datasets, _name)))
import glob
import pandas as pd
import scipy
import numpy as np
import os
from tqdm import tqdm
import pickle
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr, pearsonr
from scipy.spatial.distance import euclidean
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pickle
import glob
from collections import Counter
from pathlib import Path
from goatools.obo_parser import GODag
from goatools.anno.gaf_reader import GafReader
import glob
import pandas as pd
import scipy
import numpy as np
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.colors import ListedColormap
import textwrap
from matplotlib.colors import TwoSlopeNorm

os.makedirs("outputs/fig6_S25_S26_frangieh2021_perturbation_analysis", exist_ok=True)


In [ ]:
obodag = GODag(
    os.path.join(
        os.path.abspath("../.experiment_data/primary"), "preprocessing", "resources", "ebi_ensembl_go_mappings", "go-basic.obo"
    )
)


In [ ]:
perturbation_genes_paths_Control = glob.glob(
    "../.experiment_data/scratch/experiments_new/sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/default/calc_go_terms_perturbation_Control/expression_perturbation_perturbation_2_vs_*_Control_control_Control_res_unp_act_mask_False_flip_False_res_perturbation_perturbation_2__go_terms_evaluated_graph_csv_with_ci.pkl"
)
perturbation_genes_Control = [
    file.split("_vs_")[1].split("_Control")[0] for file in perturbation_genes_paths_Control
]
phenotypes_Control = perturbation_genes_Control
phenotypes_labels_Control = phenotypes_Control
perturbation_genes_Control


In [ ]:
perturbation_genes_paths_IFNy = glob.glob(
    "../.experiment_data/scratch/experiments_new/sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/default/calc_go_terms_perturbation_IFNγ/expression_perturbation_perturbation_2_vs_*_IFNγ_control_IFNγ_res_unp_act_mask_False_flip_False_res_perturbation_perturbation_2__go_terms_evaluated_graph_csv_with_ci.pkl"
)
perturbation_genes_IFNy = [
    file.split("_vs_")[1].split("_IFNγ")[0] for file in perturbation_genes_paths_IFNy
]
phenotypes_IFNy = perturbation_genes_IFNy
phenotypes_labels_IFNy = phenotypes_IFNy
perturbation_genes_IFNy


In [ ]:
perturbation_genes_paths_Co_culture = glob.glob(
    "../.experiment_data/scratch/experiments_new/sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/default/calc_go_terms_perturbation_Co-culture/expression_perturbation_perturbation_2_vs_*_Co-culture_control_Co-culture_res_unp_act_mask_False_flip_False_res_perturbation_perturbation_2__go_terms_evaluated_graph_csv_with_ci.pkl"
)
perturbation_genes_Co_culture = [
    file.split("_vs_")[1].split("_Co-culture")[0] for file in perturbation_genes_paths_Co_culture
]
phenotypes_Co_culture = perturbation_genes_Co_culture
phenotypes_labels_Co_culture = phenotypes_Co_culture
perturbation_genes_Co_culture


In [ ]:
perturbation_genes_paths = glob.glob(
    "../.experiment_data/scratch/experiments_new/sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/default/calc_go_terms_perturbation/expression_perturbation_vs_*_control_res_unp_act_mask_False_flip_False_res_perturbation__go_terms_evaluated_graph_csv_with_ci.pkl"
)
perturbation_genes = [
    file.split("_vs_")[1].split("_control")[0] for file in perturbation_genes_paths
]
phenotypes = perturbation_genes
phenotypes_labels = phenotypes
perturbation_genes


In [ ]:
perturbation_genes_paths_Control = sorted(perturbation_genes_paths_Control)
perturbation_genes_Control = sorted(perturbation_genes_Control)
perturbation_genes_paths_IFNy = sorted(perturbation_genes_paths_IFNy)
perturbation_genes_IFNy = sorted(perturbation_genes_IFNy)
perturbation_genes_paths_Co_culture = sorted(perturbation_genes_paths_Co_culture)
perturbation_genes_Co_culture = sorted(perturbation_genes_Co_culture)
perturbation_genes_paths = sorted(perturbation_genes_paths)
perturbation_genes = sorted(perturbation_genes)
phenotypes_labels = sorted(phenotypes_labels)


In [ ]:
import glob

def filter_gene(paths, gene_to_exclude, split_marker):
    genes = [file.split("_vs_")[1].split(split_marker)[0] for file in paths]
    filtered_genes = [gene for gene in genes if gene != gene_to_exclude]
    filtered_paths = [path for path, gene in zip(paths, genes) if gene != gene_to_exclude]
    return sorted(filtered_paths), sorted(filtered_genes)

gene_to_exclude = "CDK4"

perturbation_genes_paths_Control = glob.glob(
    "../.experiment_data/scratch/experiments_new/sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/default/calc_go_terms_perturbation_Control/expression_perturbation_perturbation_2_vs_*_Control_control_Control_res_unp_act_mask_False_flip_False_res_perturbation_perturbation_2__go_terms_evaluated_graph_csv_with_ci.pkl"
)
perturbation_genes_paths_Control, perturbation_genes_Control = filter_gene(
    perturbation_genes_paths_Control, gene_to_exclude, "_Control"
)
phenotypes_Control = perturbation_genes_Control
phenotypes_labels_Control = phenotypes_Control

perturbation_genes_paths_IFNy = glob.glob(
    "../.experiment_data/scratch/experiments_new/sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/default/calc_go_terms_perturbation_IFNγ/expression_perturbation_perturbation_2_vs_*_IFNγ_control_IFNγ_res_unp_act_mask_False_flip_False_res_perturbation_perturbation_2__go_terms_evaluated_graph_csv_with_ci.pkl"
)
perturbation_genes_paths_IFNy, perturbation_genes_IFNy = filter_gene(
    perturbation_genes_paths_IFNy, gene_to_exclude, "_IFNγ"
)
phenotypes_IFNy = perturbation_genes_IFNy
phenotypes_labels_IFNy = phenotypes_IFNy

perturbation_genes_paths_Co_culture = glob.glob(
    "../.experiment_data/scratch/experiments_new/sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/default/calc_go_terms_perturbation_Co-culture/expression_perturbation_perturbation_2_vs_*_Co-culture_control_Co-culture_res_unp_act_mask_False_flip_False_res_perturbation_perturbation_2__go_terms_evaluated_graph_csv_with_ci.pkl"
)
perturbation_genes_paths_Co_culture, perturbation_genes_Co_culture = filter_gene(
    perturbation_genes_paths_Co_culture, gene_to_exclude, "_Co-culture"
)
phenotypes_Co_culture = perturbation_genes_Co_culture
phenotypes_labels_Co_culture = phenotypes_Co_culture

perturbation_genes_paths = glob.glob(
    "../.experiment_data/scratch/experiments_new/sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/default/calc_go_terms_perturbation/expression_perturbation_vs_*_control_res_unp_act_mask_False_flip_False_res_perturbation__go_terms_evaluated_graph_csv_with_ci.pkl"
)
perturbation_genes_paths, perturbation_genes = filter_gene(
    perturbation_genes_paths, gene_to_exclude, "_control"
)
phenotypes = perturbation_genes
phenotypes_labels = phenotypes

print("Control genes:", perturbation_genes_Control)
print("IFNy genes:", perturbation_genes_IFNy)
print("Co-culture genes:", perturbation_genes_Co_culture)
print("All genes:", perturbation_genes)


In [ ]:
def aggregate_dataframes(graph_csv_list):
    num_rows = len(graph_csv_list[0])
    for df in graph_csv_list:
        if len(df) != num_rows:
            raise ValueError("All DataFrames must have the same number of rows.")

    aggregated_roc = []
    aggregated_ci = []
    aggregated_activations = []
    aggregated_act_test = []
    aggregated_case_act = []

    for row_idx in range(num_rows):
        roc_values = [df.iloc[row_idx]["predictors_label_valid_roc"] for df in graph_csv_list]
        ci_values = [
            max(
                abs(
                    df.iloc[row_idx]["predictors_label_valid_roc"]
                    - df.iloc[row_idx]["predictors_label_valid_roc_ci_upper"]
                ),
                abs(
                    df.iloc[row_idx]["predictors_label_valid_roc"]
                    - df.iloc[row_idx]["predictors_label_valid_roc_ci_lower"]
                ),
            )
            for df in graph_csv_list
        ]
        aggregated_roc.append(np.mean(roc_values))
        aggregated_ci.append(np.mean(ci_values))

        act_arrays = [df.iloc[row_idx]["activations_valid"] for df in graph_csv_list]
        concatenated = np.concatenate(act_arrays, axis=1)
        aggregated_activations.append(np.mean(concatenated, axis=1))

        aggregated_act_test.append(graph_csv_list[0].iloc[row_idx]["activations_test"])
        aggregated_case_act.append(graph_csv_list[0].iloc[row_idx]["case_activations"])

    return pd.DataFrame(
        index=graph_csv_list[0].index,
        data={
            "predictors_label_valid_roc": aggregated_roc,
            "predictors_label_valid_roc_ci": aggregated_ci,
            "activations_valid": aggregated_activations,
            "activations_test": aggregated_act_test,
            "case_activations": aggregated_case_act,
        },
    )


In [ ]:
def load_data(paths, phenotypes):
    graph_csv_list = []
    graph_csv_index_list = []
    go_importances = []
    go_ci = []
    go_activations_test = []
    go_case_activations = []

    for modality in tqdm(["expression", "protein"]):
        go_importances_modality = []
        go_ci_modality = []
        graph_csv_modality_list = []
        go_act_test_modality = []
        go_case_act_modality = []

        for path, phenotype in tqdm(zip(paths, phenotypes)):
            raw_pkl = pd.read_pickle(f"{path.replace('expression', modality)}")
            graph_csv = aggregate_dataframes([raw_pkl])

            graph_csv_modality_list.append(graph_csv)
            go_importances_modality.append(np.array(graph_csv["predictors_label_valid_roc"]))
            go_ci_modality.append(np.array(graph_csv["predictors_label_valid_roc_ci"]))
            go_act_test_modality.append(graph_csv["activations_test"].tolist())
            go_case_act_modality.append(graph_csv["case_activations"].tolist())

        graph_csv_list.append(graph_csv_modality_list)
        graph_csv_index_list.append(np.array(graph_csv_modality_list[0].index))
        go_importances.append(go_importances_modality)
        go_ci.append(go_ci_modality)
        go_activations_test.append(go_act_test_modality)
        go_case_activations.append(go_case_act_modality)

    return (
        graph_csv_list,
        graph_csv_index_list,
        go_importances,
        go_ci,
        go_activations_test,
        go_case_activations,
    )


In [ ]:
(
    graph_csv_list,
    graph_csv_index_list,
    go_importances,
    go_ci,
    go_activations_test,
    go_case_activations,
) = load_data(perturbation_genes_paths, perturbation_genes)


In [ ]:
(
    graph_csv_list_Control,
    graph_csv_index_list_Control,
    go_importances_Control,
    go_ci_Control,
    go_activations_test_Control,
    go_case_activations_Control,
) = load_data(perturbation_genes_paths_Control, perturbation_genes_Control)


In [ ]:
(
    graph_csv_list_IFNy,
    graph_csv_index_list_IFNy,
    go_importances_IFNy,
    go_ci_IFNy,
    go_activations_test_IFNy,
    go_case_activations_IFNy,
) = load_data(perturbation_genes_paths_IFNy, perturbation_genes_IFNy)


In [ ]:
(
    graph_csv_list_Co_culture,
    graph_csv_index_list_Co_culture,
    go_importances_Co_culture,
    go_ci_Co_culture,
    go_activations_test_Co_culture,
    go_case_activations_Co_culture,
) = load_data(perturbation_genes_paths_Co_culture, perturbation_genes_Co_culture)


In [ ]:
plt.hist(
    graph_csv_list_Control[0][0]["predictors_label_valid_roc"], bins=50, alpha=0.5, label="Control"
)
plt.hist(graph_csv_list_IFNy[0][0]["predictors_label_valid_roc"], bins=50, alpha=0.5, label="IFNy")
plt.hist(
    graph_csv_list_Co_culture[0][0]["predictors_label_valid_roc"],
    bins=50,
    alpha=0.5,
    label="Co-culture",
)
plt.ylabel("prevalence")
plt.xlabel("GO importance")
plt.legend()


In [ ]:
plt.hist(
    graph_csv_list_Control[1][0]["predictors_label_valid_roc"], bins=50, alpha=0.5, label="Control"
)
plt.hist(graph_csv_list_IFNy[1][0]["predictors_label_valid_roc"], bins=50, alpha=0.5, label="IFNy")
plt.hist(
    graph_csv_list_Co_culture[1][0]["predictors_label_valid_roc"],
    bins=50,
    alpha=0.5,
    label="Co-culture",
)
plt.ylabel("prevalence")
plt.xlabel("GO importance")
plt.legend()


In [ ]:
overlapping_indices = graph_csv_list[0][0].index.intersection(graph_csv_list[1][0].index)


In [ ]:
scipy.stats.spearmanr(
    graph_csv_list[0][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
    graph_csv_list[1][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
)


In [ ]:
scipy.stats.spearmanr(
    graph_csv_list_Control[0][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
    graph_csv_list_Control[1][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
)


In [ ]:
scipy.stats.spearmanr(
    graph_csv_list_IFNy[0][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
    graph_csv_list_IFNy[1][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
)


In [ ]:
scipy.stats.spearmanr(
    graph_csv_list_Co_culture[0][0]
    .loc[overlapping_indices, "predictors_label_valid_roc"]
    .to_numpy(),
    graph_csv_list_Co_culture[1][0]
    .loc[overlapping_indices, "predictors_label_valid_roc"]
    .to_numpy(),
)


In [ ]:
phenotype_index = 0


In [ ]:
go_terms_select = set(
    ["GO:0009987", "GO:0051179", "GO:0002376", "GO:0007155"]
)
for go_term in go_terms_select:
    print(obodag[go_term].name)
biological_process_children = (
    set([child.id for child in obodag["GO:0008150"].children]) - go_terms_select
)
biological_process_children.add("GO:0006810")


In [ ]:
phenotypes = ["CD58", "CD59", "ACSL3", "CDK6", "DNMT1", "ILF2", "MYC"]
bold_go_terms_labels = [
    "positive T cell selection", "negative T cell selection", "negative thymic T cell selection",
    "T cell selection", "leukocyte mediated immunity", "lymphocyte activation", "leukocyte activation",
    "leukocyte activation involved in immune response", "cell activation involved in immune response",
    "exo. antigen processing & presen.", "exo. peptide antigen processing & presen.",
    "peptide antigen processing & presen. via MHC II", "antigen processing and presentation",
]
phenotype_index = 0


### NetworkVI

In [ ]:
from matplotlib.colors import TwoSlopeNorm

def create_clustermap(data, go_phenotypes, bold_labels=None, figsize=(7, 2.4), save_name='clustermap.png'):
    vmin = 0.5 + (0.5 - data.max())
    vmax = data.max()
    center = 0.5
    norm = TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
    set_figure_params(6.45, 5)
    g = sns.clustermap(data, cmap=cmap_heat, norm=norm, figsize=figsize, cbar=True, xticklabels=go_phenotypes, yticklabels=['-IFNγ vs. control', '+IFNγ vs. control', 'TIL coculture vs. control'], col_cluster=True, row_cluster=False, cbar_kws={'orientation': 'vertical', 'fraction': 0.046, 'pad': 0.04})
    heatmap_ax = g.ax_heatmap
    fixed_height_inches = 0.3
    fig = g.fig
    fig_height_inches = fig.get_figheight()
    ax_pos = heatmap_ax.get_position()
    new_height_frac = fixed_height_inches / fig_height_inches
    heatmap_ax.set_position([ax_pos.x0, ax_pos.y0, ax_pos.width, new_height_frac])
    if bold_labels:
        for label in g.ax_heatmap.get_xticklabels():
            if label.get_text() in bold_labels:
                label.set_fontweight('bold')
            else:
                label.set_fontweight('normal')
    g.ax_col_dendrogram.set_visible(False)
    g.ax_row_dendrogram.set_visible(False)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), rotation=0, ha='left')
    g.cax.set_position([0.92, 0.42, 0.01, 0.2])
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()


In [ ]:
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import matplotlib.pyplot as plt

def create_clustermap(data, go_phenotypes, bold_labels=None, figsize=(7, 2.4), save_name='clustermap.png'):
    plot_data = np.array(data, dtype=float)
    mask_nan = np.isnan(plot_data)
    plot_data_filled = plot_data.copy()
    data_max = np.nanmax(plot_data)
    plot_data_filled[mask_nan] = 0
    vmin = 0.5 + (0.5 - data_max)
    vmax = data_max
    center = 0.5
    norm = TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
    set_figure_params(6.45, 5)
    g = sns.clustermap(plot_data_filled, cmap=cmap_heat, norm=norm, figsize=figsize, cbar=True, xticklabels=go_phenotypes, yticklabels=['-IFNγ vs. control', '+IFNγ vs. control', 'TIL coculture vs. control'], col_cluster=True, row_cluster=False, cbar_kws={'orientation': 'vertical', 'fraction': 0.046, 'pad': 0.04})
    heatmap_ax = g.ax_heatmap
    fixed_height_inches = 0.3
    fig = g.fig
    fig_height_inches = fig.get_figheight()
    ax_pos = heatmap_ax.get_position()
    new_height_frac = fixed_height_inches / fig_height_inches
    heatmap_ax.set_position([ax_pos.x0, ax_pos.y0, ax_pos.width, new_height_frac])
    if bold_labels:
        for label in g.ax_heatmap.get_xticklabels():
            if label.get_text() in bold_labels:
                label.set_fontweight('bold')
            else:
                label.set_fontweight('normal')
    g.ax_col_dendrogram.set_visible(False)
    g.ax_row_dendrogram.set_visible(False)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), rotation=0, ha='left')
    g.cax.set_position([0.92, 0.42, 0.01, 0.2])
    col_order = g.dendrogram_col.reordered_ind
    for i in range(plot_data.shape[0]):
        for j, orig_col in enumerate(col_order):
            if mask_nan[i, orig_col]:
                g.ax_heatmap.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='lightgray', edgecolor='none', zorder=3))
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

for phenotype_index, phenotype in enumerate(phenotypes):
    print("%%%%%%%%%%%%%%%%%%%%%%%")
    print(phenotype)

    graph_csv_index_list_local = [arr.copy() for arr in graph_csv_index_list]
    graph_csv_index_list_IFNy_local = [arr.copy() for arr in graph_csv_index_list_IFNy]
    graph_csv_index_list_Control_local = [arr.copy() for arr in graph_csv_index_list_Control]
    graph_csv_index_list_Co_culture_local = [arr.copy() for arr in graph_csv_index_list_Co_culture]

    go_importances_local = [arr.copy() for arr in go_importances]
    go_importances_IFNy_local = [arr.copy() for arr in go_importances_IFNy]
    go_importances_Control_local = [arr.copy() for arr in go_importances_Control]
    go_importances_Co_culture_local = [arr.copy() for arr in go_importances_Co_culture]

    for modality_index, modality in enumerate(["expression"]):
        go_terms_select_indices = [
            (
                True
                if not "UNKNOWN" in go_term
                and (
                    go_term in go_terms_select
                    or np.any(
                        [parent in go_terms_select for parent in obodag[go_term].get_all_parents()]
                    )
                )
                else False
            )
            for go_term in graph_csv_index_list_IFNy_local[modality_index]
        ]

        result_frangieh2021 = pd.DataFrame(
            index=graph_csv_index_list_local[modality_index],
            data={
                "name": [
                    obodag[go_term].name if go_term in obodag.keys() else go_term
                    for go_term in graph_csv_index_list_local[modality_index]
                ],
                "filtered": go_terms_select_indices,
            },
        )

        for go_importances_sub, go_importances_sub_name in zip(
            [
                go_importances_Control_local,
                go_importances_IFNy_local,
                go_importances_Co_culture_local,
            ],
            ["control", "IFNy", "co-culture"],
        ):
            result_frangieh2021[go_importances_sub_name] = go_importances_sub[modality_index][
                phenotype_index
            ]

        result_frangieh2021.to_csv(f"frangieh2021_further_{phenotype}_all_{modality}.csv")

    for modality_index, _ in enumerate(["expression", "protein"]):
        go_terms_select_indices = [
            (
                True
                if not "UNKNOWN" in go_term
                and (
                    go_term in go_terms_select
                    or np.any(
                        [parent in go_terms_select for parent in obodag[go_term].get_all_parents()]
                    )
                )
                else False
            )
            for go_term in graph_csv_index_list_IFNy_local[modality_index]
        ]
        print(Counter(go_terms_select_indices))

        graph_csv_index_list_local[modality_index] = graph_csv_index_list_local[modality_index][
            go_terms_select_indices
        ]
        graph_csv_index_list_IFNy_local[modality_index] = graph_csv_index_list_IFNy_local[
            modality_index
        ][go_terms_select_indices]
        graph_csv_index_list_Control_local[modality_index] = graph_csv_index_list_Control_local[
            modality_index
        ][go_terms_select_indices]
        graph_csv_index_list_Co_culture_local[modality_index] = (
            graph_csv_index_list_Co_culture_local[modality_index][go_terms_select_indices]
        )

        go_importances_local[modality_index][phenotype_index] = go_importances_local[
            modality_index
        ][phenotype_index][go_terms_select_indices]
        go_importances_IFNy_local[modality_index][phenotype_index] = go_importances_IFNy_local[
            modality_index
        ][phenotype_index][go_terms_select_indices]
        go_importances_Control_local[modality_index][phenotype_index] = (
            go_importances_Control_local[modality_index][phenotype_index][go_terms_select_indices]
        )
        go_importances_Co_culture_local[modality_index][phenotype_index] = (
            go_importances_Co_culture_local[modality_index][phenotype_index][
                go_terms_select_indices
            ]
        )

    for modality_index, modality in enumerate(["expression"]):
        threshold = 0.95 if modality == "expression" else 0.85
        indices_high_go_importances_all = set()

        for go_importances_sub in [
            go_importances_Control_local,
            go_importances_IFNy_local,
            go_importances_Co_culture_local,
        ]:
            indices_high_go_importances_sub = set(
                np.where(
                    go_importances_sub[modality_index][phenotype_index]
                    > np.quantile(go_importances_sub[modality_index][phenotype_index], threshold)
                )[0]
            )
            print(len(indices_high_go_importances_sub))
            indices_high_go_importances_all |= indices_high_go_importances_sub

        print(len(indices_high_go_importances_all))

        go_terms_labels = [
            obodag[go_term].name if go_term in obodag.keys() else go_term
            for go_term in graph_csv_index_list_local[modality_index][
                list(indices_high_go_importances_all)
            ]
        ]

        go_importances_all = np.stack(
            [
                go_importances_sub[modality_index][phenotype_index][
                    list(indices_high_go_importances_all)
                ]
                for go_importances_sub in [
                    go_importances_Control_local,
                    go_importances_IFNy_local,
                    go_importances_Co_culture_local,
                ]
            ]
        )

        go_terms_labels_dict = {
            "somatic diversification of immunoglobulins involved in immune response": "somatic diversification of Ig in immune response",
            "antigen processing and presentation of exogenous antigen": "exo. antigen processing & presen.",
            "myeloid dendritic cell antigen processing and presentation": "myeloid DC antigen processing & presen.",
            "antigen processing and presentation of exogenous peptide antigen": "exo. peptide antigen processing & presen.",
            "antigen processing and presentation of endogenous peptide antigen": "endo. peptide antigen processing & presen.",
            "antigen processing and presentation of peptide antigen via MHC class II": "peptide antigen processing & presen. via MHC II",
            "cellular response to topologically incorrect protein": "cellular response to incorrect protein folding",
            "cell-cell adhesion via plasma-membrane adhesion molecules": "cell-cell adh. via membrane adhesion molecules",
            "cellular response to oxygen-containing compound": "cell. response to oxygen-containing compound",
            "adaptive immune response based on somatic recombination of immune receptors built from immunoglobulin superfamily domains": "adaptive immunity via somatic recombination of IgSF receptors",
            "antigen processing and presentation of peptide or polysaccharide antigen via MHC class II": "peptide/polysaccharide antigen presentation via MHC II",
            "immunoglobulin production involved in immunoglobulin-mediated immune resoonse": "immunoglobulin production (Ig-mediated response)",
            "intrinsic apoptotic signaling pathway in response to DNA damage by p53 class mediator": "p53-mediated intrinsic apoptosis in response to DNA damage",
            "antimicrobial humoral immune response mediated by antimicrobial peptide": "AMP-mediated humoral immune response",
        }
        go_terms_labels = [go_terms_labels_dict.get(label, label) for label in go_terms_labels]

        print(np.max(go_importances_all))
        print(np.min(go_importances_all))

        if modality_index == 0:
            create_clustermap(
                go_importances_all,
                go_terms_labels,
                bold_labels=bold_go_terms_labels,
                save_name=f"frangieh2021_further_{phenotype}_{modality}_{mode}",
            )
        else:
            create_clustermap(
                go_importances_all,
                go_terms_labels,
                save_name=f"frangieh2021_further_{phenotype}_{modality}_{mode}",
            )


### ORA

In [ ]:
base_path = (
    "../.experiment_data/primary/"
    "sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_"
    "batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/enrichment_analysis/"
)
base_path


### GSEA

In [ ]:
base_path = (
    "../.experiment_data/primary/"
    "sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_"
    "batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/enrichment_analysis/"
)
base_path


## Fig. 6c (extended): CD58 GO-importance star plots per condition

In [ ]:
from matplotlib.colors import TwoSlopeNorm
import numpy as np
import matplotlib.pyplot as plt

def get_stars(p):
    '''* p < 0.5   (trending)'''
    if p is None or np.isnan(p):
        return ''
    if p < 0.05:
        return '***'
    elif p < 0.1:
        return '**'
    elif p < 0.5:
        return '*'
    return ''

def create_clustermap(data, go_phenotypes, pvalue_matrix=None, bold_labels=None, figsize=(7, 2.4), save_name='clustermap.png'):
    '''Parameters'''
    plot_data = np.array(data, dtype=float)
    mask_nan = np.isnan(plot_data)
    plot_data_filled = plot_data.copy()
    plot_data_filled[mask_nan] = 0
    data_max = np.nanmax(np.abs(plot_data))
    if np.nanmin(plot_data) < 0:
        vmin, vmax, center = (-data_max, data_max, 0.0)
    else:
        vmin = 0.5 + (0.5 - data_max)
        vmax = data_max
        center = 0.5
    norm = TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
    set_figure_params(6.45, 5)
    g = sns.clustermap(plot_data_filled, cmap=cmap_heat, norm=norm, figsize=figsize, cbar=True, xticklabels=go_phenotypes, yticklabels=['-IFNγ vs. control', '+IFNγ vs. control', 'TIL coculture vs. control'], col_cluster=True, row_cluster=False, cbar_kws={'orientation': 'vertical', 'fraction': 0.046, 'pad': 0.04})
    heatmap_ax = g.ax_heatmap
    fixed_height_inches = 0.3
    fig = g.fig
    fig_height_inches = fig.get_figheight()
    ax_pos = heatmap_ax.get_position()
    new_height_frac = fixed_height_inches / fig_height_inches
    heatmap_ax.set_position([ax_pos.x0, ax_pos.y0, ax_pos.width, new_height_frac])
    if bold_labels:
        for label in g.ax_heatmap.get_xticklabels():
            lw = 'bold' if label.get_text() in bold_labels else 'normal'
            label.set_fontweight(lw)
    g.ax_col_dendrogram.set_visible(False)
    g.ax_row_dendrogram.set_visible(False)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), rotation=0, ha='left')
    g.cax.set_position([0.92, 0.42, 0.01, 0.2])
    col_order = g.dendrogram_col.reordered_ind
    for i in range(plot_data.shape[0]):
        for j, orig_col in enumerate(col_order):
            if mask_nan[i, orig_col]:
                g.ax_heatmap.add_patch(plt.Rectangle((j, i), 1, 1, fill=True, color='lightgray', edgecolor='none', zorder=3))
    if pvalue_matrix is not None:
        pvalue_matrix = np.array(pvalue_matrix, dtype=float)
        for i in range(plot_data.shape[0]):
            for j, orig_col in enumerate(col_order):
                if not mask_nan[i, orig_col]:
                    stars = get_stars(pvalue_matrix[i, orig_col])
                    if stars:
                        g.ax_heatmap.text(j + 0.5, i + 0.65, stars, ha='center', va='center', fontsize=7, color='#be9e7c', fontweight='bold', zorder=4)
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()


# ORA

In [ ]:
import pandas as pd
import numpy as np
import os

base_path = (
    "../.experiment_data/primary/"
    "sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_"
    "batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/enrichment_analysis/"
)

ORA_PVAL_COL = "Adjusted P-value"

go_terms_labels_dict = {
    "somatic diversification of immunoglobulins involved in immune response": "somatic diversification of Ig in immune response",
    "antigen processing and presentation of exogenous antigen": "exo. antigen processing & presen.",
    "myeloid dendritic cell antigen processing and presentation": "myeloid DC antigen processing & presen.",
    "antigen processing and presentation of exogenous peptide antigen": "exo. peptide antigen processing & presen.",
    "antigen processing and presentation of exogenous peptide antigen via MHC class II": "exo. peptide antigen processing & presen. via MHC II",
    "antigen processing and presentation of endogenous peptide antigen": "endo. peptide antigen processing & presen.",
    "antigen processing and presentation of peptide antigen via MHC class II": "peptide antigen processing & presen. via MHC II",
    "cellular response to topologically incorrect protein": "cellular response to incorrect protein folding",
    "cell-cell adhesion via plasma-membrane adhesion molecules": "cell-cell adh. via membrane adhesion molecules",
    "cellular response to oxygen-containing compound": "cell. response to oxygen-containing compound",
    "adaptive immune response based on somatic recombination of immune receptors built from immunoglobulin superfamily domains": "adaptive immunity via somatic recombination of IgSF receptors",
    "antigen processing and presentation of peptide or polysaccharide antigen via MHC class II": "peptide/polysaccharide antigen presentation via MHC II",
    "immunoglobulin production involved in immunoglobulin-mediated immune response": "immunoglobulin production (Ig-mediated response)",
    "intrinsic apoptotic signaling pathway in response to DNA damage by p53 class mediator": "p53-mediated intrinsic apoptosis in response to DNA damage",
    "antimicrobial humoral immune response mediated by antimicrobial peptide": "AMP-mediated humoral immune response",
}

def load_go_ora(file):
    try:
        df = pd.read_csv(os.path.join(base_path, file))
        df = df[df["Gene_set"] == "GO_Biological_Process_2023"].copy()
        df["GO_ID"] = df["Term"].str.extract(r"(GO:\d+)")[0]
    except FileNotFoundError:
        df = pd.DataFrame(columns=["GO_ID", "Term", ORA_PVAL_COL])
    return df

def prepare_ora_matrix(df, score_col):
    df = df.copy()
    df["score"] = -np.log10(df[ORA_PVAL_COL] + 1e-300)
    return df[["GO_ID", "score", ORA_PVAL_COL]].rename(
        columns={"score": score_col, ORA_PVAL_COL: f"pval_{score_col}"}
    )

for perturbation_gene in phenotypes:

    go_IFNy = load_go_ora(f"go_enrichment_{perturbation_gene}_IFNy.csv")
    go_Co_culture = load_go_ora(f"go_enrichment_{perturbation_gene}_Co-culture.csv")
    go_Control = load_go_ora(f"go_enrichment_{perturbation_gene}_Control.csv")

    print(f"\n=== {perturbation_gene} ORA Overlap (all terms) ===")
    print("Control ∩ Co-culture:", len(set(go_Control["Term"]) & set(go_Co_culture["Term"])))
    print("Control ∩ IFNy:", len(set(go_Control["Term"]) & set(go_IFNy["Term"])))
    print("Co-culture ∩ IFNy:", len(set(go_Co_culture["Term"]) & set(go_IFNy["Term"])))
    print(
        "All three:",
        len(set(go_Control["Term"]) & set(go_Co_culture["Term"]) & set(go_IFNy["Term"])),
    )

    ctrl_sub = prepare_ora_matrix(go_Control, "Control")
    coc_sub = prepare_ora_matrix(go_Co_culture, "Co-culture")
    ifny_sub = prepare_ora_matrix(go_IFNy, "IFNy")

    nes = pd.merge(ctrl_sub, coc_sub, on="GO_ID", how="outer")
    nes = pd.merge(nes, ifny_sub, on="GO_ID", how="outer")
    nes = nes.set_index("GO_ID").sort_index()

    go_terms_select_indices = [
        (
            True
            if go_term in obodag.keys()
            and "UNKNOWN" not in go_term
            and (
                go_term in go_terms_select
                or np.any(
                    [parent in go_terms_select for parent in obodag[go_term].get_all_parents()]
                )
            )
            else False
        )
        for go_term in nes.index
    ]
    nes_filt = nes.iloc[go_terms_select_indices]
    print("After obodag filter:", nes_filt.shape)

    go_terms_labels = [
        obodag[go_term].name if go_term in obodag.keys() else go_term for go_term in nes_filt.index
    ]
    go_terms_labels = [go_terms_labels_dict.get(l, l) for l in go_terms_labels]

    score_cols = ["Control", "Co-culture", "IFNy"]
    pval_cols = ["pval_Control", "pval_Co-culture", "pval_IFNy"]

    data = nes_filt[score_cols].to_numpy().T
    pvalue_matrix = nes_filt[pval_cols].to_numpy().T

    try:
        create_clustermap(
            data,
            go_terms_labels,
            pvalue_matrix=pvalue_matrix,
            save_name=f"go_clustermap_ora_{perturbation_gene}",
        )
        print(f"Successful {perturbation_gene}")
    except Exception as e:
        print(f"Failed {perturbation_gene}: {e}")


# GSEA

In [ ]:
import pandas as pd
import numpy as np
import os

base_path = (
    "../.experiment_data/primary/"
    "sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_"
    "batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/enrichment_analysis/"
)

GSEA_PVAL_COL = "fdr"

go_terms_labels_dict = {
    "somatic diversification of immunoglobulins involved in immune response": "somatic diversification of Ig in immune response",
    "antigen processing and presentation of exogenous antigen": "exo. antigen processing & presen.",
    "myeloid dendritic cell antigen processing and presentation": "myeloid DC antigen processing & presen.",
    "antigen processing and presentation of exogenous peptide antigen": "exo. peptide antigen processing & presen.",
    "antigen processing and presentation of exogenous peptide antigen via MHC class II": "exo. peptide antigen processing & presen. via MHC II",
    "antigen processing and presentation of endogenous peptide antigen": "endo. peptide antigen processing & presen.",
    "antigen processing and presentation of peptide antigen via MHC class II": "peptide antigen processing & presen. via MHC II",
    "cellular response to topologically incorrect protein": "cellular response to incorrect protein folding",
    "cell-cell adhesion via plasma-membrane adhesion molecules": "cell-cell adh. via membrane adhesion molecules",
    "cellular response to oxygen-containing compound": "cell. response to oxygen-containing compound",
    "adaptive immune response based on somatic recombination of immune receptors built from immunoglobulin superfamily domains": "adaptive immunity via somatic recombination of IgSF receptors",
    "antigen processing and presentation of peptide or polysaccharide antigen via MHC class II": "peptide/polysaccharide antigen presentation via MHC II",
    "immunoglobulin production involved in immunoglobulin-mediated immune resoonse": "immunoglobulin production (Ig-mediated response)",
    "intrinsic apoptotic signaling pathway in response to DNA damage by p53 class mediator": "p53-mediated intrinsic apoptosis in response to DNA damage",
    "antimicrobial humoral immune response mediated by antimicrobial peptide": "AMP-mediated humoral immune response",
}

def load_go_gsea(file):
    try:
        df = pd.read_csv(os.path.join(base_path, file))
        df["Term"] = df["go_id"]
        df = df.rename(columns={"go_id": "GO_ID", "nes": "NES"})
    except FileNotFoundError:
        df = pd.DataFrame(columns=["GO_ID", "Term", "NES", GSEA_PVAL_COL])
    return df

def prepare_gsea_matrix(df, score_col):
    df = df.copy()
    return df[["GO_ID", "NES", GSEA_PVAL_COL]].rename(
        columns={"NES": score_col, GSEA_PVAL_COL: f"pval_{score_col}"}
    )

for perturbation_gene in phenotypes:

    go_IFNy = load_go_gsea(f"gsea_full_{perturbation_gene}_IFNy_v2.csv")
    go_Co_culture = load_go_gsea(f"gsea_full_{perturbation_gene}_Co-culture_v2.csv")
    go_Control = load_go_gsea(f"gsea_full_{perturbation_gene}_Control_v2.csv")

    print(f"\n=== {perturbation_gene} GSEA Overlap (all terms) ===")
    print("Control ∩ Co-culture:", len(set(go_Control["Term"]) & set(go_Co_culture["Term"])))
    print("Control ∩ IFNy:", len(set(go_Control["Term"]) & set(go_IFNy["Term"])))
    print("Co-culture ∩ IFNy:", len(set(go_Co_culture["Term"]) & set(go_IFNy["Term"])))
    print(
        "All three:",
        len(set(go_Control["Term"]) & set(go_Co_culture["Term"]) & set(go_IFNy["Term"])),
    )

    ctrl_sub = prepare_gsea_matrix(go_Control, "Control")
    coc_sub = prepare_gsea_matrix(go_Co_culture, "Co-culture")
    ifny_sub = prepare_gsea_matrix(go_IFNy, "IFNy")

    nes = pd.merge(ctrl_sub, coc_sub, on="GO_ID", how="outer")
    nes = pd.merge(nes, ifny_sub, on="GO_ID", how="outer")
    nes = nes.set_index("GO_ID").sort_index()

    go_terms_select_indices = [
        (
            True
            if go_term in obodag.keys()
            and "UNKNOWN" not in go_term
            and (
                go_term in go_terms_select
                or np.any(
                    [parent in go_terms_select for parent in obodag[go_term].get_all_parents()]
                )
            )
            else False
        )
        for go_term in nes.index
    ]
    nes_filt = nes.iloc[go_terms_select_indices]
    print("After obodag filter:", nes_filt.shape)

    go_terms_labels = [
        obodag[go_term].name if go_term in obodag.keys() else go_term for go_term in nes_filt.index
    ]
    go_terms_labels = [go_terms_labels_dict.get(l, l) for l in go_terms_labels]

    score_cols = ["Control", "Co-culture", "IFNy"]
    pval_cols = ["pval_Control", "pval_Co-culture", "pval_IFNy"]

    data = nes_filt[score_cols].to_numpy().T
    pvalue_matrix = nes_filt[pval_cols].to_numpy().T

    try:
        create_clustermap(
            data,
            go_terms_labels,
            pvalue_matrix=pvalue_matrix,
            save_name=f"go_clustermap_gsea_{perturbation_gene}",
        )
        print(f"Successful {perturbation_gene}")
    except Exception as e:
        print(f"Failed {perturbation_gene}: {e}")


#### Figure 5e

In [ ]:
all_gsea_scores = []
thresholds = [0.5, 0.1, 0.05]

import pandas as pd
import numpy as np
import os

base_path = (
    "../.experiment_data/primary/"
    "sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_"
    "batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/enrichment_analysis/"
)

GSEA_PVAL_COL = "fdr"

for perturbation_gene in phenotypes:
    all_gsea_scores_perturbation_gene = []

    go_IFNy = load_go_gsea(f"gsea_full_{perturbation_gene}_IFNy_v2.csv")
    go_Co_culture = load_go_gsea(f"gsea_full_{perturbation_gene}_Co-culture_v2.csv")
    go_Control = load_go_gsea(f"gsea_full_{perturbation_gene}_Control_v2.csv")

    print(f"\n=== {perturbation_gene} GSEA Overlap (all terms) ===")
    print("Control ∩ Co-culture:", len(set(go_Control["Term"]) & set(go_Co_culture["Term"])))
    print("Control ∩ IFNy:", len(set(go_Control["Term"]) & set(go_IFNy["Term"])))
    print("Co-culture ∩ IFNy:", len(set(go_Co_culture["Term"]) & set(go_IFNy["Term"])))
    print(
        "All three:",
        len(set(go_Control["Term"]) & set(go_Co_culture["Term"]) & set(go_IFNy["Term"])),
    )

    ctrl_sub = prepare_gsea_matrix(go_Control, "Control")
    coc_sub = prepare_gsea_matrix(go_Co_culture, "Co-culture")
    ifny_sub = prepare_gsea_matrix(go_IFNy, "IFNy")

    nes = pd.merge(ctrl_sub, coc_sub, on="GO_ID", how="outer")
    nes = pd.merge(nes, ifny_sub, on="GO_ID", how="outer")
    nes = nes.set_index("GO_ID").sort_index()

    go_terms_select_indices = [
        (
            True
            if go_term in obodag.keys()
            and "UNKNOWN" not in go_term
            and (
                go_term in go_terms_select
                or np.any(
                    [parent in go_terms_select for parent in obodag[go_term].get_all_parents()]
                )
            )
            else False
        )
        for go_term in nes.index
    ]
    nes_filt = nes.iloc[go_terms_select_indices]
    print("After obodag filter:", nes_filt.shape)

    go_terms_labels = [
        obodag[go_term].name if go_term in obodag.keys() else go_term for go_term in nes_filt.index
    ]
    go_terms_labels = [go_terms_labels_dict.get(l, l) for l in go_terms_labels]

    score_cols = ["Control", "Co-culture", "IFNy"]
    pval_cols = ["pval_Control", "pval_Co-culture", "pval_IFNy"]

    data = nes_filt[score_cols].to_numpy().T
    pvalue_matrix = nes_filt[pval_cols].to_numpy().T

    for score_col, pval_col in zip(score_cols, pval_cols):
        all_gsea_scores_screens = []
        num_all_go_score_col = len(nes_filt[~nes_filt[score_col].isnull()])
        all_gsea_scores_screens.append(num_all_go_score_col)
        for threshold in thresholds:
            all_gsea_scores_screens.append(
                (nes_filt[~nes_filt[score_col].isnull()][pval_col] < threshold).sum()
            )
        all_gsea_scores_perturbation_gene.append(all_gsea_scores_screens)

    all_gsea_scores.append(all_gsea_scores_perturbation_gene)


In [ ]:
import pandas as pd
import numpy as np
import os

all_ora_scores = []
thresholds = [0.5, 0.1, 0.05]

base_path = (
    "../.experiment_data/primary/"
    "sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_"
    "batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/enrichment_analysis/"
)

ORA_PVAL_COL = "Adjusted P-value"

go_terms_labels_dict = {
    "somatic diversification of immunoglobulins involved in immune response": "somatic diversification of Ig in immune response",
    "antigen processing and presentation of exogenous antigen": "exo. antigen processing & presen.",
    "myeloid dendritic cell antigen processing and presentation": "myeloid DC antigen processing & presen.",
    "antigen processing and presentation of exogenous peptide antigen": "exo. peptide antigen processing & presen.",
    "antigen processing and presentation of exogenous peptide antigen via MHC class II": "exo. peptide antigen processing & presen. via MHC II",
    "antigen processing and presentation of endogenous peptide antigen": "endo. peptide antigen processing & presen.",
    "antigen processing and presentation of peptide antigen via MHC class II": "peptide antigen processing & presen. via MHC II",
    "cellular response to topologically incorrect protein": "cellular response to incorrect protein folding",
    "cell-cell adhesion via plasma-membrane adhesion molecules": "cell-cell adh. via membrane adhesion molecules",
    "cellular response to oxygen-containing compound": "cell. response to oxygen-containing compound",
    "adaptive immune response based on somatic recombination of immune receptors built from immunoglobulin superfamily domains": "adaptive immunity via somatic recombination of IgSF receptors",
    "antigen processing and presentation of peptide or polysaccharide antigen via MHC class II": "peptide/polysaccharide antigen presentation via MHC II",
    "immunoglobulin production involved in immunoglobulin-mediated immune response": "immunoglobulin production (Ig-mediated response)",
    "intrinsic apoptotic signaling pathway in response to DNA damage by p53 class mediator": "p53-mediated intrinsic apoptosis in response to DNA damage",
    "antimicrobial humoral immune response mediated by antimicrobial peptide": "AMP-mediated humoral immune response",
}

def load_go_ora(file):
    try:
        df = pd.read_csv(os.path.join(base_path, file))
        df = df[df["Gene_set"] == "GO_Biological_Process_2023"].copy()
        df["GO_ID"] = df["Term"].str.extract(r"(GO:\d+)")[0]
    except FileNotFoundError:
        df = pd.DataFrame(columns=["GO_ID", "Term", ORA_PVAL_COL])
    return df

def prepare_ora_matrix(df, score_col):
    df = df.copy()
    df["score"] = -np.log10(df[ORA_PVAL_COL] + 1e-300)
    return df[["GO_ID", "score", ORA_PVAL_COL]].rename(
        columns={"score": score_col, ORA_PVAL_COL: f"pval_{score_col}"}
    )

for perturbation_gene in phenotypes:

    all_ora_scores_perturbation_gene = []

    go_IFNy = load_go_ora(f"go_enrichment_{perturbation_gene}_IFNy.csv")
    go_Co_culture = load_go_ora(f"go_enrichment_{perturbation_gene}_Co-culture.csv")
    go_Control = load_go_ora(f"go_enrichment_{perturbation_gene}_Control.csv")

    print(f"\n=== {perturbation_gene} ORA Overlap (all terms) ===")
    print("Control ∩ Co-culture:", len(set(go_Control["Term"]) & set(go_Co_culture["Term"])))
    print("Control ∩ IFNy:", len(set(go_Control["Term"]) & set(go_IFNy["Term"])))
    print("Co-culture ∩ IFNy:", len(set(go_Co_culture["Term"]) & set(go_IFNy["Term"])))
    print(
        "All three:",
        len(set(go_Control["Term"]) & set(go_Co_culture["Term"]) & set(go_IFNy["Term"])),
    )

    ctrl_sub = prepare_ora_matrix(go_Control, "Control")
    coc_sub = prepare_ora_matrix(go_Co_culture, "Co-culture")
    ifny_sub = prepare_ora_matrix(go_IFNy, "IFNy")

    nes = pd.merge(ctrl_sub, coc_sub, on="GO_ID", how="outer")
    nes = pd.merge(nes, ifny_sub, on="GO_ID", how="outer")
    nes = nes.set_index("GO_ID").sort_index()

    go_terms_select_indices = [
        (
            True
            if go_term in obodag.keys()
            and "UNKNOWN" not in go_term
            and (
                go_term in go_terms_select
                or np.any(
                    [parent in go_terms_select for parent in obodag[go_term].get_all_parents()]
                )
            )
            else False
        )
        for go_term in nes.index
    ]
    nes_filt = nes.iloc[go_terms_select_indices]
    print("After obodag filter:", nes_filt.shape)

    go_terms_labels = [
        obodag[go_term].name if go_term in obodag.keys() else go_term for go_term in nes_filt.index
    ]
    go_terms_labels = [go_terms_labels_dict.get(l, l) for l in go_terms_labels]

    score_cols = ["Control", "Co-culture", "IFNy"]
    pval_cols = ["pval_Control", "pval_Co-culture", "pval_IFNy"]

    for score_col, pval_col in zip(score_cols, pval_cols):
        all_ora_scores_screens = []
        num_all_go_score_col = len(nes_filt[~nes_filt[score_col].isnull()])
        all_ora_scores_screens.append(num_all_go_score_col)
        for threshold in thresholds:
            all_ora_scores_screens.append(
                (nes_filt[~nes_filt[score_col].isnull()][pval_col] < threshold).sum()
            )
        all_ora_scores_perturbation_gene.append(all_ora_scores_screens)

    all_ora_scores.append(all_ora_scores_perturbation_gene)


In [ ]:
from matplotlib.patches import Patch
cond_labels = ['Control', 'IFNγ', 'Co-culture']
cond_colors = ['#9d9c9c', '#e57373', '#00695c']
alphas = [1.0, 0.55, 0.25]
gene_order_top_to_bottom = ['ACSL3', 'CD59', 'CD58', 'DNMT1', 'CDK6', 'ILF2', 'MYC']
gene_order_bottom_to_top = list(reversed(gene_order_top_to_bottom))
gsea_order_idx = [phenotypes.index(g) for g in gene_order_bottom_to_top]
ora_genes = [i for i, gs in enumerate(all_ora_scores) if any((s[0] > 0 for s in gs))]
ora_gene_names = [phenotypes[i] for i in ora_genes]
ora_order_top_to_bottom = [g for g in gene_order_top_to_bottom if g in ora_gene_names]
ora_order_bottom_to_top = list(reversed(ora_order_top_to_bottom))
ora_order_idx = [phenotypes.index(g) for g in ora_order_bottom_to_top]
n_gsea = len(gene_order_bottom_to_top)
n_ora = len(ora_order_bottom_to_top)
bar_w = 0.22
offsets = np.array([bar_w, 0, -bar_w])
gap = 1.5
y_ora = np.arange(n_ora, dtype=float)
y_gsea = np.arange(n_gsea, dtype=float) + n_ora + gap
fig, ax = plt.subplots(figsize=(2.2, 3.1))
for ci, (color, offset) in enumerate(zip(cond_colors, offsets)):
    for thresh_idx, alpha in zip([3, 2, 1], alphas):
        heights = [all_ora_scores[ora_order_idx[gi]][ci][thresh_idx] for gi in range(n_ora)]
        ax.barh(y_ora + offset, heights, height=bar_w, color=color, alpha=alpha, zorder=2)
for ci, (color, offset) in enumerate(zip(cond_colors, offsets)):
    for thresh_idx, alpha in zip([3, 2, 1], alphas):
        heights = [all_gsea_scores[gsea_order_idx[gi]][ci][thresh_idx] for gi in range(n_gsea)]
        ax.barh(y_gsea + offset, heights, height=bar_w, color=color, alpha=alpha, zorder=2, label=cond_labels[ci] if thresh_idx == 3 else '')
y_sep = (y_ora[-1] + y_gsea[0]) / 2
ax.axhline(y_sep, -0.5, ls=':', color='grey', lw=1.2, zorder=3, clip_on=False)
ax.set_yticks(np.concatenate([y_ora, y_gsea]))
ax.set_yticklabels(ora_order_bottom_to_top + gene_order_bottom_to_top)
bracket_x = -14

def draw_bracket(ax, y_min, y_max, x, label):
    ax.plot([x, x], [y_min - 0.35, y_max + 0.35], color='black', lw=0.7, clip_on=False)
    ax.plot([x, x + 0.6], [y_max + 0.35, y_max + 0.35], color='black', lw=0.7, clip_on=False)
    ax.plot([x, x + 0.6], [y_min - 0.35, y_min - 0.35], color='black', lw=0.7, clip_on=False)
    ax.text(x - 0.5, (y_min + y_max) / 2, label, ha='right', va='center', fontsize=5, rotation=90, clip_on=False)
draw_bracket(ax, y_ora[0], y_ora[-1], bracket_x - 10, 'ORA')
draw_bracket(ax, y_gsea[0], y_gsea[-1], bracket_x - 10, 'GSEA')
ax.set_xlabel('n sig. GO terms')
ax.spines[['top', 'right']].set_visible(False)
ax.set_xlim(left=0)
fig.subplots_adjust(left=0.32)
leg_screen = ax.legend(fontsize=4, handlelength=1, framealpha=0, loc='upper left', bbox_to_anchor=(1.02, 1.0), title='Screen', title_fontsize=4)
ax.add_artist(leg_screen)
fdr_legend = [Patch(facecolor='grey', alpha=1.0, label='< 0.05'), Patch(facecolor='grey', alpha=0.55, label='< 0.10'), Patch(facecolor='grey', alpha=0.25, label='< 0.50')]
ax.legend(handles=fdr_legend, fontsize=4, handlelength=1, framealpha=0, loc='upper left', bbox_to_anchor=(1.02, 0.82), title='FDR / Adj. P-value', title_fontsize=4)
plt.savefig(os.path.join(_OUTDIR_ABS, 'frangieh2021_gsea_ora_unified.png'), dpi=150, bbox_inches='tight', transparent=True)
plt.savefig(os.path.join(_OUTDIR_ABS, 'frangieh2021_gsea_ora_unified.svg'), dpi=150, bbox_inches='tight', transparent=True, format='svg')
plt.savefig(os.path.join(_OUTDIR_ABS, 'frangieh2021_gsea_ora_unified.pdf'), bbox_inches='tight', transparent=True)
plt.savefig(os.path.join(_OUTDIR_ABS, 'frangieh2021_gsea_ora_unified.svg'), bbox_inches='tight', transparent=True, format='svg')
plt.show()


In [ ]:
from matplotlib.patches import Patch
cond_labels = ['Control', 'IFNγ', 'Co-culture']
cond_colors = ['#9d9c9c', '#e57373', '#00695c']
alphas = [1.0, 0.55, 0.25]
gene_order_top_to_bottom = ['ACSL3', 'CD59', 'CD58', 'DNMT1', 'CDK6', 'ILF2', 'MYC']
gene_order_bottom_to_top = list(reversed(gene_order_top_to_bottom))
gsea_order_idx = [phenotypes.index(g) for g in gene_order_bottom_to_top]
ora_genes = [i for i, gs in enumerate(all_ora_scores) if any((s[0] > 0 for s in gs))]
ora_gene_names = [phenotypes[i] for i in ora_genes]
ora_order_top_to_bottom = [g for g in gene_order_top_to_bottom if g in ora_gene_names]
ora_order_bottom_to_top = list(reversed(ora_order_top_to_bottom))
ora_order_idx = [phenotypes.index(g) for g in ora_order_bottom_to_top]
n_gsea = len(gene_order_bottom_to_top)
n_ora = len(ora_order_bottom_to_top)
bar_w = 0.22
offsets = np.array([bar_w, 0, -bar_w])
gap = 1.5
y_ora = np.arange(n_ora, dtype=float)
y_gsea = np.arange(n_gsea, dtype=float) + n_ora + gap
fig, ax = plt.subplots(figsize=(2.2, 3.1))
for ci, (color, offset) in enumerate(zip(cond_colors, offsets)):
    for thresh_idx, alpha in zip([3, 2, 1], alphas):
        heights = [all_ora_scores[ora_order_idx[gi]][ci][thresh_idx] for gi in range(n_ora)]
        ax.barh(y_ora + offset, heights, height=bar_w, color=color, alpha=alpha, zorder=2)
for ci, (color, offset) in enumerate(zip(cond_colors, offsets)):
    for thresh_idx, alpha in zip([3, 2, 1], alphas):
        heights = [all_gsea_scores[gsea_order_idx[gi]][ci][thresh_idx] for gi in range(n_gsea)]
        ax.barh(y_gsea + offset, heights, height=bar_w, color=color, alpha=alpha, zorder=2, label=cond_labels[ci] if thresh_idx == 3 else '')
y_sep = (y_ora[-1] + y_gsea[0]) / 2
ax.axhline(y_sep, -0.5, ls=':', color='grey', lw=1.2, zorder=3, clip_on=False)
ax.set_yticks(np.concatenate([y_ora, y_gsea]))
ax.set_yticklabels(ora_order_bottom_to_top + gene_order_bottom_to_top)
bracket_x = -14

def draw_bracket(ax, y_min, y_max, x, label):
    ax.plot([x, x], [y_min - 0.35, y_max + 0.35], color='black', lw=0.7, clip_on=False)
    ax.plot([x, x + 0.6], [y_max + 0.35, y_max + 0.35], color='black', lw=0.7, clip_on=False)
    ax.plot([x, x + 0.6], [y_min - 0.35, y_min - 0.35], color='black', lw=0.7, clip_on=False)
    ax.text(x - 0.5, (y_min + y_max) / 2, label, ha='right', va='center', fontsize=5, rotation=90, clip_on=False)
draw_bracket(ax, y_ora[0], y_ora[-1], bracket_x - 10, 'ORA')
draw_bracket(ax, y_gsea[0], y_gsea[-1], bracket_x - 10, 'GSEA')
ax.set_xlabel('n sig. GO terms')
ax.spines[['top', 'right']].set_visible(False)
ax.set_xlim(left=0)
fig.subplots_adjust(left=0.32)
leg_screen = ax.legend(fontsize=4, handlelength=1, framealpha=0, loc='upper left', bbox_to_anchor=(1.02, 1.0), title='Screen', title_fontsize=4)
ax.add_artist(leg_screen)
fdr_legend = [Patch(facecolor='grey', alpha=1.0, label='< 0.05'), Patch(facecolor='grey', alpha=0.55, label='< 0.10'), Patch(facecolor='grey', alpha=0.25, label='< 0.50')]
ax.legend(handles=fdr_legend, fontsize=4, handlelength=1, framealpha=0, loc='upper left', bbox_to_anchor=(1.02, 0.82), title='FDR / Adj. P-value', title_fontsize=4)
plt.savefig(os.path.join(_OUTDIR_ABS, 'frangieh2021_gsea_ora_unified.png'), dpi=150, bbox_inches='tight', transparent=True)
plt.savefig(os.path.join(_OUTDIR_ABS, 'frangieh2021_gsea_ora_unified.svg'), dpi=150, bbox_inches='tight', transparent=True, format='svg')
plt.savefig(os.path.join(_OUTDIR_ABS, 'frangieh2021_gsea_ora_unified.pdf'), bbox_inches='tight', transparent=True)
plt.savefig(os.path.join(_OUTDIR_ABS, 'frangieh2021_gsea_ora_unified.svg'), bbox_inches='tight', transparent=True, format='svg')
plt.show()


In [ ]:
from matplotlib.colors import TwoSlopeNorm

def create_clustermap(data, go_phenotypes, bold_labels=None, figsize=(7, 2.4), save_name='clustermap.png'):
    vmin = 0.5 + (0.5 - data.max())
    vmax = data.max()
    center = 0.5
    norm = TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
    set_figure_params(6.45, 5)
    g = sns.clustermap(data, cmap=cmap_heat, norm=norm, figsize=figsize, cbar=True, xticklabels=go_phenotypes, yticklabels=['-IFNγ vs. control', '+IFNγ vs. control', 'TIL coculture vs. control'], col_cluster=True, row_cluster=False, cbar_kws={'orientation': 'vertical', 'fraction': 0.046, 'pad': 0.04})
    heatmap_ax = g.ax_heatmap
    fixed_height_inches = 0.3
    fig = g.fig
    fig_height_inches = fig.get_figheight()
    ax_pos = heatmap_ax.get_position()
    new_height_frac = fixed_height_inches / fig_height_inches
    heatmap_ax.set_position([ax_pos.x0, ax_pos.y0, ax_pos.width, new_height_frac])
    if bold_labels:
        for label in g.ax_heatmap.get_xticklabels():
            if label.get_text() in bold_labels:
                label.set_fontweight('bold')
            else:
                label.set_fontweight('normal')
    g.ax_col_dendrogram.set_visible(False)
    g.ax_row_dendrogram.set_visible(False)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), rotation=0, ha='left')
    g.cax.set_position([0.92, 0.42, 0.01, 0.2])
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.png'), dpi=500, format='png', transparent=True, bbox_inches='tight')
        plt.savefig(os.path.join(_OUTDIR_ABS, f'{save_name}.svg'), dpi=500, transparent=True, bbox_inches='tight', format='svg')
    else:
        plt.show()


In [ ]:
import numpy as np
import pandas as pd
from collections import Counter

for phenotype_index, phenotype in enumerate(phenotypes):
    print("%%%%%%%%%%%%%%%%%%%%%%%")
    print(phenotype)

    # per-array gene index
    _idx_main = perturbation_genes.index(phenotype)
    _idx_control = perturbation_genes_Control.index(phenotype)
    _idx_ifny = perturbation_genes_IFNy.index(phenotype)
    _idx_coculture = perturbation_genes_Co_culture.index(phenotype)

    graph_csv_index_list_local = [arr.copy() for arr in graph_csv_index_list]
    graph_csv_index_list_IFNy_local = [arr.copy() for arr in graph_csv_index_list_IFNy]
    graph_csv_index_list_Control_local = [arr.copy() for arr in graph_csv_index_list_Control]
    graph_csv_index_list_Co_culture_local = [arr.copy() for arr in graph_csv_index_list_Co_culture]

    go_importances_local = [arr.copy() for arr in go_importances]
    go_importances_IFNy_local = [arr.copy() for arr in go_importances_IFNy]
    go_importances_Control_local = [arr.copy() for arr in go_importances_Control]
    go_importances_Co_culture_local = [arr.copy() for arr in go_importances_Co_culture]

    for modality_index, modality in enumerate(["expression"]):
        go_terms_select_indices = [
            (
                True
                if not "UNKNOWN" in go_term
                and (
                    go_term in go_terms_select
                    or np.any(
                        [parent in go_terms_select for parent in obodag[go_term].get_all_parents()]
                    )
                )
                else False
            )
            for go_term in graph_csv_index_list_IFNy_local[modality_index]
        ]

        result_frangieh2021 = pd.DataFrame(
            index=graph_csv_index_list_local[modality_index],
            data={
                "name": [
                    obodag[go_term].name if go_term in obodag.keys() else go_term
                    for go_term in graph_csv_index_list_local[modality_index]
                ],
                "filtered": go_terms_select_indices,
            },
        )

        for go_importances_sub, go_importances_sub_name, _idx in zip(
            [
                go_importances_Control_local,
                go_importances_IFNy_local,
                go_importances_Co_culture_local,
            ],
            ["control", "IFNy", "co-culture"],
            [_idx_control, _idx_ifny, _idx_coculture],
        ):
            result_frangieh2021[go_importances_sub_name] = go_importances_sub[modality_index][
                _idx
            ]

        result_frangieh2021.to_csv(f"frangieh2021_further_{phenotype}_all_{modality}.csv")

    for modality_index, _ in enumerate(["expression", "protein"]):
        go_terms_select_indices = [
            (
                True
                if not "UNKNOWN" in go_term
                and (
                    go_term in go_terms_select
                    or np.any(
                        [parent in go_terms_select for parent in obodag[go_term].get_all_parents()]
                    )
                )
                else False
            )
            for go_term in graph_csv_index_list_IFNy_local[modality_index]
        ]
        print(Counter(go_terms_select_indices))

        graph_csv_index_list_local[modality_index] = graph_csv_index_list_local[modality_index][
            go_terms_select_indices
        ]
        graph_csv_index_list_IFNy_local[modality_index] = graph_csv_index_list_IFNy_local[
            modality_index
        ][go_terms_select_indices]
        graph_csv_index_list_Control_local[modality_index] = graph_csv_index_list_Control_local[
            modality_index
        ][go_terms_select_indices]
        graph_csv_index_list_Co_culture_local[modality_index] = (
            graph_csv_index_list_Co_culture_local[modality_index][go_terms_select_indices]
        )

        go_importances_local[modality_index][_idx_main] = go_importances_local[
            modality_index
        ][_idx_main][go_terms_select_indices]
        go_importances_IFNy_local[modality_index][_idx_ifny] = go_importances_IFNy_local[
            modality_index
        ][_idx_ifny][go_terms_select_indices]
        go_importances_Control_local[modality_index][_idx_control] = (
            go_importances_Control_local[modality_index][_idx_control][go_terms_select_indices]
        )
        go_importances_Co_culture_local[modality_index][_idx_coculture] = (
            go_importances_Co_culture_local[modality_index][_idx_coculture][
                go_terms_select_indices
            ]
        )

    for modality_index, modality in enumerate(["expression"]):
        threshold = 0.95 if modality == "expression" else 0.85
        indices_high_go_importances_all = set()

        for go_importances_sub in [
            go_importances_Co_culture_local
        ]:
            indices_high_go_importances_sub = set(
                np.where(
                    go_importances_sub[modality_index][_idx_coculture]
                    > np.quantile(go_importances_sub[modality_index][_idx_coculture], threshold)
                )[0]
            )
            print(len(indices_high_go_importances_sub))
            indices_high_go_importances_all |= indices_high_go_importances_sub

        print(len(indices_high_go_importances_all))

        go_terms_labels = [
            obodag[go_term].name if go_term in obodag.keys() else go_term
            for go_term in graph_csv_index_list_local[modality_index][
                list(indices_high_go_importances_all)
            ]
        ]

        go_importances_all = np.stack(
            [
                go_importances_sub[modality_index][_idx][
                    list(indices_high_go_importances_all)
                ]
                for go_importances_sub, _idx in [
                    (go_importances_Control_local, _idx_control),
                    (go_importances_IFNy_local, _idx_ifny),
                    (go_importances_Co_culture_local, _idx_coculture),
                ]
            ]
        )

        go_terms_labels_dict = {
            "somatic diversification of immunoglobulins involved in immune response": "somatic diversification of Ig in immune response",
            "antigen processing and presentation of exogenous antigen": "exo. antigen processing & presen.",
            "myeloid dendritic cell antigen processing and presentation": "myeloid DC antigen processing & presen.",
            "antigen processing and presentation of exogenous peptide antigen": "exo. peptide antigen processing & presen.",
            "antigen processing and presentation of peptide antigen via MHC class II": "peptide antigen processing & presen. via MHC II",
            "cellular response to topologically incorrect protein": "cellular response to incorrect protein folding",
            "cell-cell adhesion via plasma-membrane adhesion molecules": "cell-cell adh. via membrane adhesion molecules",
            "cellular response to oxygen-containing compound": "cell. response to oxygen-containing compound",
        }
        go_terms_labels = [go_terms_labels_dict.get(label, label) for label in go_terms_labels]

        print(np.max(go_importances_all))
        print(np.min(go_importances_all))

        if modality_index == 0:
            create_clustermap(
                go_importances_all,
                go_terms_labels,
                bold_labels=bold_go_terms_labels,
                save_name=f"frangieh2021_further_{phenotype}_{modality}_{mode}",
            )
        else:
            create_clustermap(
                go_importances_all,
                go_terms_labels,
                save_name=f"frangieh2021_further_{phenotype}_{modality}_{mode}",
            )


## Fig. 6c: CD58 GO importance (exact original selection/threshold)

In [ ]:
bold_go_terms_labels_cd58_original = [
    "positive T cell selection", "negative T cell selection", "negative thymic T cell selection",
    "T cell selection", "leukocyte mediated immunity", "lymphocyte activation", "leukocyte activation",
    "leukocyte activation involved in immune response", "cell activation involved in immune response",
    "exo. antigen processing & presen.", "exo. peptide antigen processing & presen.",
    "peptide antigen processing & presen. via MHC II", "antigen processing and presentation",
]

def create_clustermap_cd58_original(data, go_phenotypes, bold_labels=None, figsize=(7, 2.4), save_name=False):
    vmin = 0.5 + (0.5 - data.max())
    vmax = data.max()
    center = 0.5
    norm = TwoSlopeNorm(vmin=vmin, vcenter=center, vmax=vmax)
    set_figure_params(6.45, 2.4)
    g = sns.clustermap(
        data, cmap=cmap_heat, norm=norm, figsize=figsize, cbar=True,
        xticklabels=go_phenotypes,
        yticklabels=["-IFNγ vs. control", "+IFNγ vs. control", "TIL coculture vs. control"],
        col_cluster=True, row_cluster=True,
        cbar_kws={"orientation": "vertical", "fraction": 0.046, "pad": 0.04},
    )
    if bold_labels:
        for label in g.ax_heatmap.get_xticklabels():
            label.set_fontweight("bold" if label.get_text() in bold_labels else "normal")
    g.ax_col_dendrogram.set_visible(False)
    g.ax_row_dendrogram.set_visible(False)
    g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), rotation=0, ha="left")
    g.cax.set_position([0.92, 0.42, 0.01, 0.2])
    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f"{save_name}.png"), dpi=500, format="png", transparent=True, bbox_inches="tight")
        plt.savefig(os.path.join(_OUTDIR_ABS, f"{save_name}.svg"), dpi=500, transparent=True, bbox_inches="tight", format="svg")
    else:
        plt.show()

# GO-term pre-filter
_go_terms_select_cd58 = set(["GO:0050896", "GO:0002376", "GO:0007155", "GO:0006955"])

_idx_control = perturbation_genes_Control.index("CD58")
_idx_ifny = perturbation_genes_IFNy.index("CD58")
_idx_coculture = perturbation_genes_Co_culture.index("CD58")

graph_csv_index_list_cd58_local = [arr.copy() for arr in graph_csv_index_list]
go_importances_Control_cd58_local = [arr.copy() for arr in go_importances_Control]
go_importances_IFNy_cd58_local = [arr.copy() for arr in go_importances_IFNy]
go_importances_Co_culture_cd58_local = [arr.copy() for arr in go_importances_Co_culture]

for modality_index, modality in enumerate(["expression", "protein"]):
    go_terms_select_indices_cd58 = [
        (
            True
            if not "UNKNOWN" in go_term
            and (
                go_term in _go_terms_select_cd58
                or np.any(
                    [parent in _go_terms_select_cd58 for parent in obodag[go_term].get_all_parents()]
                )
            )
            else False
        )
        for go_term in graph_csv_index_list_IFNy[modality_index]
    ]
    graph_csv_index_list_cd58_local[modality_index] = graph_csv_index_list_cd58_local[modality_index][
        go_terms_select_indices_cd58
    ]
    go_importances_Control_cd58_local[modality_index] = go_importances_Control_cd58_local[modality_index][
        _idx_control
    ][go_terms_select_indices_cd58]
    go_importances_IFNy_cd58_local[modality_index] = go_importances_IFNy_cd58_local[modality_index][
        _idx_ifny
    ][go_terms_select_indices_cd58]
    go_importances_Co_culture_cd58_local[modality_index] = go_importances_Co_culture_cd58_local[modality_index][
        _idx_coculture
    ][go_terms_select_indices_cd58]

for modality_index, modality in enumerate(["expression", "protein"]):
    threshold = 0.95 if modality == "expression" else 0.85
    indices_high_go_importances_all = set()
    for go_importances_sub in [
        go_importances_Control_cd58_local,
        go_importances_IFNy_cd58_local,
        go_importances_Co_culture_cd58_local,
    ]:
        indices_high_go_importances_sub = set(
            np.where(
                go_importances_sub[modality_index]
                > np.quantile(go_importances_sub[modality_index], threshold)
            )[0]
        )
        indices_high_go_importances_all = indices_high_go_importances_all | indices_high_go_importances_sub

    go_terms_labels = [
        obodag[go_term].name if go_term in obodag.keys() else go_term
        for go_term in graph_csv_index_list_cd58_local[modality_index][list(indices_high_go_importances_all)]
    ]
    go_importances_all = np.stack([
        go_importances_sub[modality_index][list(indices_high_go_importances_all)]
        for go_importances_sub in [
            go_importances_Control_cd58_local,
            go_importances_IFNy_cd58_local,
            go_importances_Co_culture_cd58_local,
        ]
    ])

    go_terms_labels_dict = {
        "somatic diversification of immunoglobulins involved in immune response": "somatic diversification of Ig in immune response",
        "antigen processing and presentation of exogenous antigen": "exo. antigen processing & presen.",
        "myeloid dendritic cell antigen processing and presentation": "myeloid DC antigen processing & presen.",
        "antigen processing and presentation of exogenous peptide antigen": "exo. peptide antigen processing & presen.",
        "antigen processing and presentation of peptide antigen via MHC class II": "peptide antigen processing & presen. via MHC II",
        "cellular response to topologically incorrect protein": "cellular response to incorrect protein folding",
        "cell-cell adhesion via plasma-membrane adhesion molecules": "cell-cell adh. via membrane adhesion molecules",
        "cellular response to oxygen-containing compound": "cell. response to oxygen-containing compound",
    }
    go_terms_labels = [go_terms_labels_dict.get(label, label) for label in go_terms_labels]

    if modality_index == 0:
        create_clustermap_cd58_original(
            go_importances_all, go_terms_labels,
            bold_labels=bold_go_terms_labels_cd58_original,
            save_name=f"frangieh2021_CD58_{modality}_{mode}",
        )
    else:
        create_clustermap_cd58_original(
            go_importances_all, go_terms_labels,
            save_name=f"frangieh2021_CD58_{modality}_{mode}",
        )


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from scipy.stats import norm

def _build_labels(activations_test, case_activations):
    '''Reconstruct the binary label vector the same way get_ci_auroc does:'''
    case_set = set(map(tuple, case_activations))
    return np.array(
        [tuple(row) in case_set for row in activations_test], dtype=int
    )

def _logreg_scores(activations_test, labels):
    """Fit logistic regression and return predicted probabilities (same as get_ci_auroc)."""
    clf = LogisticRegression(
        solver="liblinear", class_weight="balanced", max_iter=100
    )
    clf.fit(activations_test, labels)
    return clf.predict_proba(activations_test)[:, 1]

def _auroc_var_from_scores(y_score, y_true):
    '''Hanley–McNeil V-statistic variance of AUROC.'''
    pos = y_score[y_true == 1]
    neg = y_score[y_true == 0]
    n_p, n_n = len(pos), len(neg)
    if n_p == 0 or n_n == 0:
        return 0.5, 0.0

    V10 = (pos[:, None] > neg[None, :]).mean(axis=1) + \
          0.5 * (pos[:, None] == neg[None, :]).mean(axis=1)
    V01 = (neg[:, None] < pos[None, :]).mean(axis=1) + \
          0.5 * (neg[:, None] == pos[None, :]).mean(axis=1)

    auc = V10.mean()
    var = np.var(V10, ddof=1) / n_p + np.var(V01, ddof=1) / n_n
    return auc, var

def two_sample_delong(act_a, case_a, act_b, case_b):
    '''Two-sided two-sample DeLong test comparing AUROCs between two screens.'''
    lbl_a   = _build_labels(act_a, case_a)
    lbl_b   = _build_labels(act_b, case_b)
    score_a = _logreg_scores(act_a, lbl_a)
    score_b = _logreg_scores(act_b, lbl_b)

    auc_a, var_a = _auroc_var_from_scores(score_a, lbl_a)
    auc_b, var_b = _auroc_var_from_scores(score_b, lbl_b)

    var_diff = var_a + var_b
    if var_diff <= 0:
        return 0.0, 1.0

    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2 * norm.sf(abs(z))
    return z, p


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
alpha = 0.05
corrected_alpha = alpha / 3
c_dark = '#003754'
c_light = '#c5dce3'

def two_sample_delong_from_ci(auc_a, ci_a, auc_b, ci_b):
    se_a = ci_a / 1.96
    se_b = ci_b / 1.96
    var_diff = se_a ** 2 + se_b ** 2
    if var_diff <= 0:
        return (0.0, 1.0)
    z = (auc_a - auc_b) / np.sqrt(var_diff)
    p = 2 * norm.sf(abs(z))
    return (z, p)
records = []
for phenotype_index, phenotype in enumerate(phenotypes):
    graph_csv_index_list_local = [arr.copy() for arr in graph_csv_index_list]
    ctrl = go_importances_Control[0][phenotype_index]
    ifny = go_importances_IFNy[0][phenotype_index]
    til = go_importances_Co_culture[0][phenotype_index]
    ci_c = go_ci_Control[0][phenotype_index]
    ci_i = go_ci_IFNy[0][phenotype_index]
    ci_t = go_ci_Co_culture[0][phenotype_index]
    go_terms_select_indices = np.array([True if 'UNKNOWN' not in go_term and (go_term in go_terms_select or np.any([parent in go_terms_select for parent in obodag[go_term].get_all_parents()])) else False for go_term in graph_csv_index_list_local[0]])
    sel = np.where(go_terms_select_indices)[0]
    n_total = len(sel)
    ctrl = ctrl[sel]
    ifny = ifny[sel]
    til = til[sel]
    ci_c = ci_c[sel]
    ci_i = ci_i[sel]
    ci_t = ci_t[sel]
    diff_mask = np.zeros(n_total, dtype=bool)
    for i in range(n_total):
        pairs = [(ctrl[i], ci_c[i], ifny[i], ci_i[i]), (ctrl[i], ci_c[i], til[i], ci_t[i]), (ifny[i], ci_i[i], til[i], ci_t[i])]
        for auc_a, ci_a, auc_b, ci_b in pairs:
            _, p = two_sample_delong_from_ci(auc_a, ci_a, auc_b, ci_b)
            if p < corrected_alpha:
                diff_mask[i] = True
                break
    above_mask = (ctrl > 0.53) | (ifny > 0.53) | (til > 0.53)
    n_above_only = int((above_mask & ~diff_mask).sum())
    n_diff = int((above_mask & diff_mask).sum())
    records.append({'phenotype': phenotype, 'n_above_only': n_above_only, 'n_diff': n_diff, 'n_total_above': n_above_only + n_diff, 'n_total': n_total})
records_sorted = sorted(records, key=lambda r: r['n_total_above'])
phen_order = [r['phenotype'] for r in records_sorted]
n_above_only = [r['n_above_only'] for r in records_sorted]
n_diff = [r['n_diff'] for r in records_sorted]
n_total = records_sorted[0]['n_total']
y = np.arange(len(phen_order))
figure_width, figure_height = (1.4, 1.8)
set_figure_params(figure_width, figure_height)
fig, ax = plt.subplots(figsize=(figure_width, figure_height))
ax.barh(y, n_above_only, color=c_light, label='AUROC > 0.53 in ≥1 screen')
ax.barh(y, n_diff, left=n_above_only, color=c_dark, label='+ diff. across screens\n(DeLong, Bonferroni)')
ax.axvline(n_total, ls='--', label='Total modelled', color='gray', lw=0.8)
ax.set_yticks(y)
ax.set_yticklabels(phen_order)
ax.set_xlabel('Number of GO terms')
ax.invert_yaxis()
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=4, handlelength=1, framealpha=0, loc='lower right', bbox_to_anchor=(1, 1))
plt.tight_layout()
plt.savefig(os.path.join(_OUTDIR_ABS, 'differential_GO_terms_horizontal.png'), dpi=500, bbox_inches='tight', transparent=True)
plt.savefig(os.path.join(_OUTDIR_ABS, 'differential_GO_terms_horizontal.svg'), dpi=500, bbox_inches='tight', transparent=True, format='svg')
plt.show()


## Fig. 6a, S25a: UMAP of the latent space (unperturbed vs. CD58-perturbed cells)

In [ ]:
import matplotlib.colors as mcolors

def lighten_color(color, amount=0.6):
    c = np.array(mcolors.to_rgba(color))
    white = np.array([1, 1, 1, 1])
    return tuple((1 - amount) * c + amount * white)

colors_encoded_dict = {
    "control": {
        "IFNγ": lighten_color("#4c626c", 0.3),
        "Co-culture": lighten_color("#00695c", 0.3),
        "Control": lighten_color("#e57373", 0.3),
    },
    "CD58": {
        "IFNγ": lighten_color("#70acc0", 0),
        "Co-culture": lighten_color("#70acc0", 0),
        "Control": lighten_color("#70acc0", 0),
    },
    "MYC": {
        "IFNγ": lighten_color("#c12075", 0),
        "Co-culture": lighten_color("#c12075", 0),
        "Control": lighten_color("#c12075", 0),
    },
    "CD274": {
        "IFNγ": lighten_color("#be9e7c", 0),
        "Co-culture": lighten_color("#be9e7c", 0),
        "Control": lighten_color("#be9e7c", 0),
    },
}

def plot_perturbation_umap(X_umap, labels, perturbation_2, colors_encoded_dict, save_name=None):
    set_figure_params(4, 4)
    fig, ax = plt.subplots()
    ax.set_aspect("equal", "box")

    colors_encoded = [
        colors_encoded_dict[label][per_2]
        for label, per_2 in zip(labels[np.isin(labels, ["control"])], perturbation_2[np.isin(labels, ["control"])])
    ]
    ax.scatter(
        X_umap[np.isin(labels, ["control"])][:, 0],
        X_umap[np.isin(labels, ["control"])][:, 1],
        s=0.8,
        c=colors_encoded,
        edgecolor="none",
    )
    for perturbation in ["CD58"]:
        colors_encoded = [
            colors_encoded_dict[label][per_2]
            for label, per_2 in zip(
                labels[np.isin(labels, [perturbation])], perturbation_2[np.isin(labels, [perturbation])]
            )
        ]
        ax.scatter(
            X_umap[np.isin(labels, [perturbation])][:, 0],
            X_umap[np.isin(labels, [perturbation])][:, 1],
            s=3,
            c=colors_encoded,
            edgecolor="none",
            marker="^",
        )

    ax.set_xticks([])
    ax.set_yticks([])
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_box_aspect(1)

    if save_name:
        plt.savefig(os.path.join(_OUTDIR_ABS, f"{save_name}.png"), dpi=500, format="png", transparent=True, bbox_inches="tight")
        plt.savefig(os.path.join(_OUTDIR_ABS, f"{save_name}.svg"), dpi=500, transparent=True, bbox_inches="tight", format="svg")
    else:
        plt.show()
    plt.close()


In [ ]:
_frangieh_networkvi_dir = (
    "../.experiment_data/primary/experiments_new/"
    "sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_"
    "batcov_False_patcov_False_modtype_go_with_gene_layer_gaf_standard_with_rna_s0/default"
)
X_umap = np.load(os.path.join(_frangieh_networkvi_dir, "X_umap_with_perturbation.npy"))
perturbation_2 = np.load(os.path.join(_frangieh_networkvi_dir, "perturbation_2_with_perturbation.npy"), allow_pickle=True)
labels = np.load(os.path.join(_frangieh_networkvi_dir, "labels_with_perturbation.npy"), allow_pickle=True)

plot_perturbation_umap(X_umap, labels, perturbation_2, colors_encoded_dict, save_name="frangieh2021_sparsevi_cd58_perturbation_umap")


In [ ]:
_frangieh_multivi_dir = (
    "../.experiment_data/primary/experiments_new/"
    "sparsevi_frangieh2021_1.0_scib_hvar_True_batscib_False_patscib_False_mmixTrue_no_gating_"
    "batcov_False_patcov_False_modtype_dense_s0/default"
)
X_umap = np.load(os.path.join(_frangieh_multivi_dir, "X_umap_with_perturbation.npy"))
perturbation_2 = np.load(os.path.join(_frangieh_multivi_dir, "perturbation_2_with_perturbation.npy"), allow_pickle=True)
labels = np.load(os.path.join(_frangieh_multivi_dir, "labels_with_perturbation.npy"), allow_pickle=True)

plot_perturbation_umap(X_umap, labels, perturbation_2, colors_encoded_dict, save_name="frangieh2021_multivi_cd58_perturbation_umap")


## Superseded (not used)

In [ ]:
overlapping_indices = graph_csv_list[0][0][
    graph_csv_list[0][0]["predictors_label_valid_roc"] > 0.52
].index.intersection(
    graph_csv_list[1][0][graph_csv_list[1][0]["predictors_label_valid_roc"] > 0.52].index
)
scipy.stats.spearmanr(
    graph_csv_list[0][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
    graph_csv_list[1][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
)


In [ ]:
overlapping_indices = graph_csv_list_Control[0][0][
    graph_csv_list_Control[0][0]["predictors_label_valid_roc"] > 0.5
].index.intersection(
    graph_csv_list_Control[1][0][
        graph_csv_list_Control[1][0]["predictors_label_valid_roc"] > 0.5
    ].index
)
print(len(overlapping_indices))
scipy.stats.spearmanr(
    graph_csv_list_Control[0][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
    graph_csv_list_Control[1][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
)


In [ ]:
overlapping_indices = graph_csv_list_IFNy[0][0][
    graph_csv_list_IFNy[0][0]["predictors_label_valid_roc"] > 0.52
].index.intersection(
    graph_csv_list_IFNy[1][0][graph_csv_list_IFNy[1][0]["predictors_label_valid_roc"] > 0.52].index
)
scipy.stats.spearmanr(
    graph_csv_list_IFNy[0][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
    graph_csv_list_IFNy[1][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
)


In [ ]:
overlapping_indices = graph_csv_list_Co_culture[0][0][
    graph_csv_list_Co_culture[0][0]["predictors_label_valid_roc"] > 0.51
].index.intersection(
    graph_csv_list_Co_culture[1][0][
        graph_csv_list_Co_culture[1][0]["predictors_label_valid_roc"] > 0.51
    ].index
)
print(len(overlapping_indices))
scipy.stats.spearmanr(
    graph_csv_list_Co_culture[0][0]
    .loc[overlapping_indices, "predictors_label_valid_roc"]
    .to_numpy(),
    graph_csv_list_Co_culture[1][0]
    .loc[overlapping_indices, "predictors_label_valid_roc"]
    .to_numpy(),
)


In [ ]:
overlapping_indices = graph_csv_list_Co_culture[0][0][
    graph_csv_list_Co_culture[0][0]["predictors_label_valid_roc"] > 0.5
].index.intersection(
    graph_csv_list_IFNy[0][0][graph_csv_list_IFNy[0][0]["predictors_label_valid_roc"] > 0.5].index
)
print(len(overlapping_indices))
scipy.stats.spearmanr(
    graph_csv_list_Co_culture[0][0]
    .loc[overlapping_indices, "predictors_label_valid_roc"]
    .to_numpy(),
    graph_csv_list_IFNy[0][0].loc[overlapping_indices, "predictors_label_valid_roc"].to_numpy(),
)


In [ ]:
def get_rank(graph_csv_list_sub, go_term):
    sorted_list = sorted(graph_csv_list_sub["predictors_label_valid_roc"], reverse=True)
    value = graph_csv_list_sub.loc[go_term, "predictors_label_valid_roc"]
    rank = sorted_list.index(value) + 1
    return rank, value

for go_term in [
    "GO:0042288",
    "GO:0002592",
    "GO:0019886",
    "GO:0002577",
    "GO:0002580",
    "GO:0002592",
    "GO:0019883",
    "GO:0002495",
    "GO:0002478",
    "GO:0002504",
    "GO:0019882",
    "GO:0048002",
    "GO:0002577",
    "GO:0002583",
    "GO:0002580",
    "GO:0019884",
    "GO:0019882",
]:
    if go_term in graph_csv_list_Co_culture[0][0].index:
        print(go_term)
        print(obodag[go_term].name)
        print("expression")
        print(get_rank(graph_csv_list_Control[0][0], go_term))
        print(get_rank(graph_csv_list_IFNy[0][0], go_term))
        print(get_rank(graph_csv_list_Co_culture[0][0], go_term))
    if go_term in graph_csv_list_Co_culture[1][0].index:
        print(go_term)
        print(obodag[go_term].name)
        print("protein")
        print(get_rank(graph_csv_list_Control[1][0], go_term))
        print(get_rank(graph_csv_list_IFNy[1][0], go_term))
        print(get_rank(graph_csv_list_Co_culture[1][0], go_term))


In [ ]:
for modality_index, modality in enumerate(["expression", "protein"]):
    go_terms_select_indices = [
        (
            True
            if not "UNKNOWN" in go_term
            and (
                go_term in go_terms_select
                or np.any(
                    [
                        True if parent in go_terms_select else False
                        for parent in obodag[go_term].get_all_parents()
                    ]
                )
            )
            else False
        )
        for go_term in graph_csv_index_list_IFNy[modality_index]
    ]
    result_frangieh2021 = pd.DataFrame(
        index=graph_csv_index_list[modality_index],
        data={
            "name": [
                obodag[go_term].name if go_term in obodag.keys() else go_term
                for go_term in graph_csv_index_list[modality_index]
            ],
            "filtered": go_terms_select_indices,
        },
    )
    for go_importances_sub, go_importances_sub_name in zip(
        [go_importances_Control, go_importances_IFNy, go_importances_Co_culture],
        ["control", "IFNy", "co-culture"],
    ):
        result_frangieh2021[go_importances_sub_name] = go_importances_sub[modality_index][
            phenotype_index
        ]
    result_frangieh2021.to_csv(f"frangieh2021_further_{phenotype}_all_{modality}.csv")


In [ ]:
for modality_index, _ in enumerate(["expression", "protein"]):
    go_terms_select_indices = [
        (
            True
            if not "UNKNOWN" in go_term
            and (
                go_term in go_terms_select
                or np.any(
                    [
                        True if parent in go_terms_select else False
                        for parent in obodag[go_term].get_all_parents()
                    ]
                )
            )
            else False
        )
        for go_term in graph_csv_index_list_IFNy[modality_index]
    ]
    print(Counter(go_terms_select_indices))
    graph_csv_index_list[modality_index] = graph_csv_index_list[modality_index][
        go_terms_select_indices
    ]
    graph_csv_index_list_IFNy[modality_index] = graph_csv_index_list_IFNy[modality_index][
        go_terms_select_indices
    ]
    graph_csv_index_list_Control[modality_index] = graph_csv_index_list_Control[modality_index][
        go_terms_select_indices
    ]
    graph_csv_index_list_Co_culture[modality_index] = graph_csv_index_list_Co_culture[
        modality_index
    ][go_terms_select_indices]
    go_importances[modality_index][phenotype_index] = go_importances[modality_index][
        phenotype_index
    ][go_terms_select_indices]
    go_importances_IFNy[modality_index][phenotype_index] = go_importances_IFNy[modality_index][
        phenotype_index
    ][go_terms_select_indices]
    go_importances_Control[modality_index][phenotype_index] = go_importances_Control[
        modality_index
    ][phenotype_index][go_terms_select_indices]
    go_importances_Co_culture[modality_index][phenotype_index] = go_importances_Co_culture[
        modality_index
    ][phenotype_index][go_terms_select_indices]


In [ ]:
for modality_index, modality in enumerate(["expression", "protein"]):
    threshold = 0.95
    if modality == "expression":
        threshold = 0.95
    else:
        threshold = 0.85
    indices_high_go_importances_all = set()
    for go_importances_sub in [
        go_importances_Control,
        go_importances_IFNy,
        go_importances_Co_culture,
    ]:
        indices_high_go_importances_sub = set(
            np.where(
                go_importances_sub[modality_index][phenotype_index]
                > np.quantile(go_importances_sub[modality_index][phenotype_index], threshold)
            )[0]
        )
        print(len(indices_high_go_importances_sub))
        indices_high_go_importances_all = (
            indices_high_go_importances_all | indices_high_go_importances_sub
        )

    print(len(indices_high_go_importances_all))
    print(len(indices_high_go_importances_all))

    go_terms_labels = [
        obodag[go_term].name if go_term in obodag.keys() else go_term
        for go_term in graph_csv_index_list[modality_index][list(indices_high_go_importances_all)]
    ]

    go_importances_all = np.stack(
        [
            go_importances_sub[modality_index][phenotype_index][
                list(indices_high_go_importances_all)
            ]
            for go_importances_sub in [
                go_importances_Control,
                go_importances_IFNy,
                go_importances_Co_culture,
            ]
        ]
    )

    print([go_term_label for go_term_label in go_terms_labels if len(go_term_label) > 50])
    go_terms_labels_dict = {
        "somatic diversification of immunoglobulins involved in immune response": "somatic diversification of Ig in immune response",
        "antigen processing and presentation of exogenous antigen": "exo. antigen processing & presen.",
        "myeloid dendritic cell antigen processing and presentation": "myeloid DC antigen processing & presen.",
        "antigen processing and presentation of exogenous peptide antigen": "exo. peptide antigen processing & presen.",
        "antigen processing and presentation of peptide antigen via MHC class II": "peptide antigen processing & presen. via MHC II",
        "cellular response to topologically incorrect protein": "cellular response to incorrect protein folding",
        "cell-cell adhesion via plasma-membrane adhesion molecules": "cell-cell adh. via membrane adhesion molecules",
        "cellular response to oxygen-containing compound": "cell. response to oxygen-containing compound",
    }

    go_terms_labels = [
        go_terms_labels_dict.get(go_term_label, go_term_label) for go_term_label in go_terms_labels
    ]

    print(np.max(go_importances_all))
    print(np.min(go_importances_all))
    print(go_importances_all.max())
    print(go_importances_all.min())
    if modality_index == 0:
        create_clustermap(
            go_importances_all,
            go_terms_labels,
            bold_labels=bold_go_terms_labels,
            save_name=f"frangieh2021_further_{phenotype}_{modality}_{mode}",
        )
    else:
        create_clustermap(
            go_importances_all,
            go_terms_labels,
            save_name=f"frangieh2021_further_{phenotype}_{modality}_{mode}",
        )
